# 03 · Skew & Term Structure Analysis

Full smile analytics from the calibrated SVI surface:
- 25Δ and 10Δ Risk Reversals and Butterflies across all expiries
- ATM vol term structure and forward vol curve bootstrap
- Skew stickiness ratio
- Risk-neutral density (Breeden-Litzenberger)
- Realised vol cone and VRP estimation

**Run notebooks 01 and 02 first.**

**Outputs:** `results/skew_dashboard.png`

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from src.surface_fit import VolSurface
from src.skew_analyzer import SkewAnalyzer
from src.term_structure import TermStructureAnalyzer
from src.deribit_client import DeribitClient

Path('../results').mkdir(exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 130,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
print('Libraries loaded.')

In [ ]:
files = sorted(Path('../data/raw').glob('btc_surface_*.parquet'))
if not files:
    raise FileNotFoundError(
        'No BTC surface files found. Run notebook 01_data_collection.ipynb first.'
    )

snap = pd.read_parquet(files[-1])
print(f'Loaded: {files[-1].name}  ({len(snap)} options, {snap["expiry"].nunique()} expiries)')

surface = VolSurface(min_strikes=5)
surface.fit(snap, iv_col='calc_iv')
print(f'Surface fitted: {len(surface.slices)} slices')

In [ ]:
sa   = SkewAnalyzer(surface)
tsa  = TermStructureAnalyzer(surface)
F_ref = surface.slices[0].F

print('=== SKEW SUMMARY TABLE ===')
skew_summ = sa.summary_table()
display(skew_summ[[
    'expiry', 'T_days', 'atm_vol_pct',
    'rr_25d_pct', 'bf_25d_pct',
    'rr_10d_pct', 'bf_10d_pct',
    'skew_slope', 'ssr'
]].style.format({
    'T_days': '{:.0f}',
    'atm_vol_pct': '{:.2f}',
    'rr_25d_pct': '{:+.2f}',
    'bf_25d_pct': '{:+.2f}',
    'rr_10d_pct': '{:+.2f}',
    'bf_10d_pct': '{:+.2f}',
    'skew_slope': '{:.4f}',
    'ssr': '{:.3f}',
}))

In [ ]:
ts_df = tsa.atm_term_structure()

print('=== ATM TERM STRUCTURE ===')
display(ts_df[['expiry', 'T_days', 'atm_vol_pct', 'fwd_vol_pct']].style.format({
    'T_days': '{:.0f}',
    'atm_vol_pct': '{:.2f}',
    'fwd_vol_pct': '{:.2f}',
}))

print('\n=== TERM STRUCTURE SUMMARY ===')
for k, v in tsa.summary(F=F_ref).items():
    print(f'  {k:<25} {v}')

In [ ]:
fig = plt.figure(figsize=(16, 14))
gs  = gridspec.GridSpec(3, 2, hspace=0.50, wspace=0.35)

T_days = skew_summ['T_days'].values

# 1. ATM vol term structure
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(ts_df['T_days'], ts_df['atm_vol_pct'], 'o-',
         color='#2E86AB', lw=2.5, ms=8, zorder=5, label='ATM IV')
ax1.plot(ts_df['T_days'], ts_df['fwd_vol_pct'], 's--',
         color='#E84855', lw=2, ms=7, zorder=4, label='Forward vol')
ax1.set_xlabel('Days to expiry')
ax1.set_ylabel('Vol (%)')
ax1.set_title('ATM Vol & Forward Vol Term Structure', fontweight='bold')
ax1.legend()

# 2. Risk Reversal term structure
ax2 = fig.add_subplot(gs[0, 1])
colors_rr = ['#E84855' if v < 0 else '#2E86AB' for v in skew_summ['rr_25d_pct']]
ax2.bar(T_days, skew_summ['rr_25d_pct'], color=colors_rr, alpha=0.85, label='25Δ RR')
ax2.plot(T_days, skew_summ['rr_10d_pct'], 's--',
         color='#F6AE2D', lw=2, ms=7, label='10Δ RR')
ax2.axhline(0, color='black', lw=0.8)
ax2.set_xlabel('Days to expiry')
ax2.set_ylabel('Vol pts (%)')
ax2.set_title('Risk Reversal Term Structure', fontweight='bold')
ax2.legend()

# 3. Butterfly term structure
ax3 = fig.add_subplot(gs[1, 0])
ax3.plot(T_days, skew_summ['bf_25d_pct'], 'o-',
         color='#F18F01', lw=2.5, ms=8, label='25Δ BF')
ax3.plot(T_days, skew_summ['bf_10d_pct'], 's--',
         color='#6B4226', lw=2, ms=7, label='10Δ BF')
ax3.set_xlabel('Days to expiry')
ax3.set_ylabel('Vol pts (%)')
ax3.set_title('Butterfly Term Structure', fontweight='bold')
ax3.legend()

# 4. ATM skew slope
ax4 = fig.add_subplot(gs[1, 1])
ax4.plot(T_days, skew_summ['skew_slope'] * 100, 'o-',
         color='#5C4742', lw=2.5, ms=8)
ax4.axhline(0, color='black', lw=0.8)
ax4.set_xlabel('Days to expiry')
ax4.set_ylabel('∂σ/∂k (vol pts)')
ax4.set_title('ATM Skew Slope ∂σ/∂k', fontweight='bold')

# 5. Full smile in delta space (nearest expiry)
ax5 = fig.add_subplot(gs[2, 0])
first_exp = surface.slices[0].expiry_str
smile_df  = sa.smile_grid(first_exp, n_points=120)
ax5.plot(smile_df['delta'] * 100, smile_df['iv_pct'],
         color='#2E86AB', lw=2.5)
for d_pct in [10, 25, 50, 75, 90]:
    ax5.axvline(d_pct, color='grey', lw=0.7, ls='--', alpha=0.6)
    ax5.text(d_pct, smile_df['iv_pct'].max() * 1.01, f'{d_pct}Δ',
             ha='center', fontsize=7, color='grey')
ax5.set_xlabel('Delta (%)')
ax5.set_ylabel('IV (%)')
ax5.set_title(f'Smile in Delta Space — {first_exp}', fontweight='bold')

# 6. Risk-neutral density
ax6 = fig.add_subplot(gs[2, 1])
rnd = sa.risk_neutral_density(first_exp, n_points=200)
ax6.fill_between(rnd['K'], rnd['density'], alpha=0.35, color='#E84855')
ax6.plot(rnd['K'], rnd['density'], color='#E84855', lw=2)
ax6.axvline(surface.slices[0].F, color='navy', ls='--', lw=1.5, label='Forward (ATM)')
ax6.set_xlabel('Strike ($)')
ax6.set_ylabel('Risk-neutral density')
ax6.set_title('Risk-Neutral Density (Breeden-Litzenberger)', fontweight='bold')
ax6.legend()

plt.suptitle(
    f'BTC Options — Skew & Term Structure  |  Spot ${snap["spot"].iloc[0]:,.0f}',
    fontsize=14, fontweight='bold', y=1.01
)
plt.savefig('../results/skew_dashboard.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved results/skew_dashboard.png')

In [ ]:
print('=== FORWARD VOL CURVE ===')
fv_df = tsa.forward_vol_curve(F=F_ref)
display(fv_df.style.format({
    'T1_days': '{:.0f}',
    'T2_days': '{:.0f}',
    'fwd_vol': '{:.4f}',
    'fwd_vol_pct': '{:.2f}',
}))

In [ ]:
print('=== DVOL HISTORY (BTC) ===')
try:
    client = DeribitClient()
    dvol   = client.get_historical_volatility('BTC')
    print(f'  Retrieved {len(dvol)} daily DVOL readings')
    print(f'  Latest: {dvol.iloc[-1]:.2f}%  ({dvol.index[-1].date()})')
    print(f'  30d avg: {dvol.tail(30).mean():.2f}%')
    print(f'  90d avg: {dvol.tail(90).mean():.2f}%')

    fig2, ax = plt.subplots(figsize=(12, 4))
    dvol.tail(180).plot(ax=ax, color='#2E86AB', lw=1.5)
    ax.set_ylabel('DVOL (%)')
    ax.set_title('BTC DVOL — Last 180 Days', fontweight='bold')
    plt.tight_layout()
    plt.savefig('../results/dvol_history.png', bbox_inches='tight', dpi=130)
    plt.show()
except Exception as e:
    print(f'  DVOL fetch failed: {e}')